# ScamSense — 03: Dataset Merge & Train/Val/Test Split

**Purpose:** Load all raw datasets, standardise to a common schema, merge into one dataframe, clean, balance classes, and produce stratified train/val/test splits ready for model training.

**Input:** `data/raw/` — 6 phishing CSVs, NUS SMS JSON, UCI SMS, job postings CSV, Wikipedia CSVs, synthetic scam CSV  
**Output:** `data/processed/train.csv`, `val.csv`, `test.csv`

| Cell | Description |
|---|---|
| 1 | Imports and paths |
| 2 | Load scam sources (label=1) |
| 3 | Load ham sources (label=0) |
| 4 | Merge and clean |
| 5 | Class balance check and undersampling |
| 6 | Train / val / test split |
| 7 | Save and verify |

**Class balance strategy:** undersample ham to match scam count per language.  
Rationale: simpler to explain in FPR, produces cleaner per-class evaluation metrics.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =============================================================================
# CELL 1: Imports and path config
# =============================================================================
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# All paths relative to repo root (set by notebook 01)
RAW       = "/content/drive/MyDrive/ScamSense/data/raw"
PROCESSED = "/content/drive/MyDrive/ScamSense/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

# Target split ratios
VAL_RATIO  = 0.15
TEST_RATIO = 0.15  # 70 / 15 / 15

RANDOM_STATE = 42
print("Config ready.")

Config ready.


In [3]:
import os, shutil

src = '/content/drive/MyDrive/ScamSense/data/processed/scamsense_synthetic_scam_only.csv'
dst = '/content/drive/MyDrive/ScamSense/data/raw/synthetic/scamsense_synthetic_scam_only.csv'

os.makedirs(os.path.dirname(dst), exist_ok=True)

if os.path.exists(src):
    shutil.copy(src, dst)
    print(f"✓ Copied to {dst}")
else:
    # Check what's actually in processed/
    processed = '/content/drive/MyDrive/ScamSense/data/processed'
    print("Files in processed/:")
    for f in os.listdir(processed):
        size = os.path.getsize(f'{processed}/{f}') / 1e6
        print(f'  {f} ({size:.1f} MB)')

✓ Copied to /content/drive/MyDrive/ScamSense/data/raw/synthetic/scamsense_synthetic_scam_only.csv


In [4]:
# =============================================================================
# CELL 2: Load all scam sources (label = 1)
# -----------------------------------------------------------------------------
# Sources:
#   A. Synthetic scam data (notebook 02 output) — 5 languages, 20,426 rows
#   B. Phishing Email Dataset — 6 CSV files, extract phishing rows only
#   C. UCI SMS Spam Collection — extract spam rows only
#   D. Real/Fake Job Postings — extract fraudulent postings only
#
# All sources standardised to: text | label | language | source
# =============================================================================

scam_frames = []

# --- A. Synthetic scam data (already label=1, has language column) ---
print("Loading synthetic scam data...")
df_synthetic = pd.read_csv(
    f"{RAW}/synthetic/scamsense_synthetic_scam_only.csv",
    encoding="utf-8-sig",
)
df_synthetic = df_synthetic[["text", "label", "language"]].copy()
df_synthetic["source"] = "synthetic_spf2025"
scam_frames.append(df_synthetic)
print(f"  ✓ Synthetic: {len(df_synthetic):,} rows")

# --- B. Phishing Email Dataset (6 CSV files) ---
# Column names vary by file — we normalise them.
# All files have a text/body column and a label column (1=phishing, 0=legit).
print("Loading Phishing Email Dataset...")
df_phishing = pd.read_csv(
    f"{RAW}/phishing/phishing_email.csv",
    usecols=["text_combined", "label"],
    on_bad_lines="skip",
    encoding="utf-8",
)
df_phishing.columns = ["text", "label"]
df_phishing["language"] = "en"
df_phishing["source"] = "phishing_email"

df_phishing_scam = df_phishing[df_phishing["label"] == 1].copy()
df_phishing_ham  = df_phishing[df_phishing["label"] == 0].copy()

scam_frames.append(df_phishing_scam)
print(f"  ✓ Phishing scam rows: {len(df_phishing_scam):,}")
print(f"  ✓ Phishing ham rows:  {len(df_phishing_ham):,}  (added to ham in Cell 3)")

# --- C. UCI SMS Spam Collection ---
# Tab-separated, no header. Column 0 = 'ham'/'spam', Column 1 = text.
print("Loading UCI SMS spam rows...")
df_uci = pd.read_csv(
    f"{RAW}/uci_sms/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label_str", "text"],
    on_bad_lines="skip",
)
df_uci_spam = df_uci[df_uci["label_str"] == "spam"].copy()
df_uci_spam["label"] = 1
df_uci_spam["language"] = "en"
df_uci_spam["source"] = "uci_sms_spam"
df_uci_spam = df_uci_spam[["text", "label", "language", "source"]]
scam_frames.append(df_uci_spam)
print(f"  ✓ UCI SMS spam: {len(df_uci_spam):,} rows")

# --- D. Real/Fake Job Postings ---
# fraudulent column: 1 = fake job post (scam), 0 = real
print("Loading fake job postings...")
df_jobs = pd.read_csv(f"{RAW}/job_postings/fake_job_postings.csv", on_bad_lines="skip")
# Combine title + description as the text field for richer signal
df_jobs["text"] = (
    df_jobs["title"].fillna("") + " " +
    df_jobs["description"].fillna("")
).str.strip()
df_jobs_scam = df_jobs[df_jobs["fraudulent"] == 1].copy()
df_jobs_scam["label"] = 1
df_jobs_scam["language"] = "en"
df_jobs_scam["source"] = "job_postings_fake"
df_jobs_scam = df_jobs_scam[["text", "label", "language", "source"]]
scam_frames.append(df_jobs_scam)
print(f"  ✓ Fake job postings: {len(df_jobs_scam):,} rows")

# --- Merge all scam sources ---
df_scam = pd.concat(scam_frames, ignore_index=True)
print(f"\nTotal scam rows (before cleaning): {len(df_scam):,}")
print(df_scam["source"].value_counts())

Loading synthetic scam data...
  ✓ Synthetic: 20,482 rows
Loading Phishing Email Dataset...
  ✓ Phishing scam rows: 42,891
  ✓ Phishing ham rows:  39,595  (added to ham in Cell 3)
Loading UCI SMS spam rows...
  ✓ UCI SMS spam: 747 rows
Loading fake job postings...
  ✓ Fake job postings: 866 rows

Total scam rows (before cleaning): 64,986
source
phishing_email       42891
synthetic_spf2025    20482
job_postings_fake      866
uci_sms_spam           747
Name: count, dtype: int64


In [5]:
# =============================================================================
# CELL 3: Load all ham sources (label = 0)
# -----------------------------------------------------------------------------
# Sources:
#   A. Phishing Email Dataset — legitimate email rows (label=0)
#   B. UCI SMS Spam Collection — ham SMS rows
#   C. NUS SMS Corpus — real Singapore SMS messages (Singlish + Mandarin)
#   D. Real/Fake Job Postings — real job postings
#   E. Wikipedia paragraphs — Malay, Tamil, Mandarin neutral text
#
# Wikipedia paragraphs serve as non-English ham for MS/TA/ZH languages.
# NUS SMS Corpus provides real Singlish and Mandarin conversational ham.
# =============================================================================

ham_frames = []

# --- A. Phishing Email Dataset — legitimate rows ---
print("Loading phishing email ham rows...")
ham_frames.append(df_phishing_ham)
print(f"Phishing ham: {len(df_phishing_ham):,} rows")

# --- B. UCI SMS ham ---
print("Loading UCI SMS ham rows...")
df_uci_ham = df_uci[df_uci["label_str"] == "ham"].copy()
df_uci_ham["label"] = 0
df_uci_ham["language"] = "en"
df_uci_ham["source"] = "uci_sms_ham"
df_uci_ham = df_uci_ham[["text", "label", "language", "source"]]
ham_frames.append(df_uci_ham)
print(f"  ✓ UCI SMS ham: {len(df_uci_ham):,} rows")

# --- C. NUS SMS Corpus ---
# JSON format: list of objects with 'talkFrom' and 'talkBody' fields.
# Two files: English/Singlish and Mandarin.
print("Loading NUS SMS Corpus...")
NUS_FILES = [
    ("smsCorpus_en_2015.03.09_all.json",  "singlish"),
    ("smsCorpus_zh_2015.03.09.json",       "zh"),
]
for fname, lang in NUS_FILES:
    fpath = f"{RAW}/nus_sms/{fname}"
    if not os.path.exists(fpath):
        print(f"  ⚠ Skipped (not found): {fname}")
        continue
    with open(fpath, encoding="utf-8") as f:
        raw = json.load(f)
    # NUS corpus structure: {"smsCorpus": {"message": [{"text": {"$": "..."}}]}}
    try:
        messages = raw["smsCorpus"]["message"]
        texts = [m["text"]["$"] for m in messages if "$" in m.get("text", {})]
    except (KeyError, TypeError):
        # Fallback: flat list of strings
        texts = [str(m) for m in raw if isinstance(m, str)]
    df_nus = pd.DataFrame({"text": texts, "label": 0, "language": lang, "source": "nus_sms"})
    ham_frames.append(df_nus)
    print(f"  ✓ NUS SMS ({lang}): {len(df_nus):,} rows")

# --- D. Real job postings ---
print("Loading real job postings...")
df_jobs_ham = df_jobs[df_jobs["fraudulent"] == 0].copy()
df_jobs_ham["text"] = (
    df_jobs_ham["title"].fillna("") + " " +
    df_jobs_ham["description"].fillna("")
).str.strip()
df_jobs_ham["label"] = 0
df_jobs_ham["language"] = "en"
df_jobs_ham["source"] = "job_postings_real"
df_jobs_ham = df_jobs_ham[["text", "label", "language", "source"]]
ham_frames.append(df_jobs_ham)
print(f"  ✓ Real job postings: {len(df_jobs_ham):,} rows")

# --- E. Wikipedia multilingual paragraphs ---
# Used as non-English ham for Malay, Tamil, Mandarin.
# We use the 'text' column which contains article paragraphs.
print("Loading Wikipedia multilingual ham...")
WIKI_FILES = [
    ("wiki_ms.csv", "ms"),
    ("wiki_ta.csv", "ta"),
    ("wiki_zh.csv", "zh"),
]
for fname, lang in WIKI_FILES:
    fpath = f"{RAW}/wikipedia/{fname}"
    if not os.path.exists(fpath):
        print(f"  ⚠ Skipped (not found): {fname}")
        continue
    df_wiki = pd.read_csv(fpath, usecols=["text"], on_bad_lines="skip")
    df_wiki["label"] = 0
    df_wiki["language"] = lang
    df_wiki["source"] = f"wikipedia_{lang}"
    ham_frames.append(df_wiki)
    print(f"  ✓ Wikipedia {lang}: {len(df_wiki):,} rows")

# --- Merge all ham sources ---
df_ham = pd.concat(ham_frames, ignore_index=True)
print(f"\nTotal ham rows (before cleaning): {len(df_ham):,}")
print(df_ham["source"].value_counts())

Loading phishing email ham rows...
Phishing ham: 39,595 rows
Loading UCI SMS ham rows...
  ✓ UCI SMS ham: 4,825 rows
Loading NUS SMS Corpus...
  ✓ NUS SMS (singlish): 55,835 rows
  ✓ NUS SMS (zh): 31,460 rows
Loading real job postings...
  ✓ Real job postings: 17,014 rows
Loading Wikipedia multilingual ham...
  ✓ Wikipedia ms: 5,000 rows
  ✓ Wikipedia ta: 5,000 rows
  ✓ Wikipedia zh: 5,000 rows

Total ham rows (before cleaning): 163,729
source
nus_sms              87295
phishing_email       39595
job_postings_real    17014
wikipedia_ta          5000
wikipedia_ms          5000
wikipedia_zh          5000
uci_sms_ham           4825
Name: count, dtype: int64


In [6]:
# =============================================================================
# CELL 4: Merge all sources and clean
# -----------------------------------------------------------------------------
# Cleaning steps applied to merged dataframe:
#   1. Drop rows with null or empty text
#   2. Strip leading/trailing whitespace
#   3. Drop texts shorter than 5 characters (noise / encoding artifacts)
#   4. Deduplicate on (text, language) — same text in different languages is valid
#   5. Reset index
# =============================================================================

print("Merging scam and ham...")
df_all = pd.concat([df_scam, df_ham], ignore_index=True)
print(f"Raw merged shape: {df_all.shape}")

# Ensure consistent column types
df_all["text"]     = df_all["text"].astype(str)
df_all["label"]    = df_all["label"].astype(int)
df_all["language"] = df_all["language"].astype(str)
df_all["source"]   = df_all["source"].astype(str)

# Step 1: Drop nulls
before = len(df_all)
df_all = df_all.dropna(subset=["text"])
print(f"After drop nulls: {len(df_all):,} (removed {before - len(df_all):,})")

# Step 2: Strip whitespace
df_all["text"] = df_all["text"].str.strip()

# Step 3: Drop very short texts (under 5 chars — encoding artifacts or empty)
before = len(df_all)
df_all = df_all[df_all["text"].str.len() >= 5]
print(f"After drop short texts: {len(df_all):,} (removed {before - len(df_all):,})")

# Step 4: Deduplicate on (text, language)
before = len(df_all)
df_all = df_all.drop_duplicates(subset=["text", "language"]).reset_index(drop=True)
print(f"After dedup: {len(df_all):,} (removed {before - len(df_all):,})")

print("\n--- Label distribution after cleaning ---")
print(df_all["label"].value_counts())
print("\n--- Language distribution after cleaning ---")
print(df_all["language"].value_counts())
print("\n--- Per-language label breakdown ---")
print(df_all.groupby(["language", "label"]).size().unstack(fill_value=0))

Merging scam and ham...
Raw merged shape: (228715, 4)
After drop nulls: 228,715 (removed 0)
After drop short texts: 221,771 (removed 6,944)
After dedup: 212,122 (removed 9,649)

--- Label distribution after cleaning ---
label
0    147555
1     64567
Name: count, dtype: int64

--- Language distribution after cleaning ---
language
en          107920
singlish     51661
zh           34685
ta            9758
ms            8098
Name: count, dtype: int64

--- Per-language label breakdown ---
label         0      1
language              
en        58765  49155
ms         4974   3124
singlish  47957   3704
ta         4997   4761
zh        30862   3823


In [7]:
# =============================================================================
# CELL 5: Class balance check and undersampling
# -----------------------------------------------------------------------------
# Strategy: undersample ham (label=0) to match scam (label=1) count per language.
#
# Why per-language rather than globally?
# Some languages (en) have many more ham rows than others (ms, ta).
# Per-language undersampling ensures each language is individually balanced,
# preventing English ham from dominating the final dataset.
#
# Languages with fewer ham rows than scam rows are kept as-is (no upsampling).
# =============================================================================

print("Balancing classes per language...\n")
balanced_frames = []

for lang in df_all["language"].unique():
    df_lang = df_all[df_all["language"] == lang]
    scam_count = len(df_lang[df_lang["label"] == 1])
    ham_count  = len(df_lang[df_lang["label"] == 0])

    df_scam_lang = df_lang[df_lang["label"] == 1]
    df_ham_lang  = df_lang[df_lang["label"] == 0]

    if ham_count > scam_count:
        # Undersample ham to match scam count
        df_ham_lang = df_ham_lang.sample(n=scam_count, random_state=RANDOM_STATE)
        print(f"{lang}: scam={scam_count:,} | ham undersampled {ham_count:,} -> {scam_count:,}")
    elif ham_count < scam_count:
        # Keep all ham — do not oversample
        print(f"{lang}: scam={scam_count:,} | ham={ham_count:,} (ham short — keeping all)")
    else:
        print(f"{lang}: scam={scam_count:,} | ham={ham_count:,} (already balanced)")

    balanced_frames.append(df_scam_lang)
    balanced_frames.append(df_ham_lang)

df_balanced = pd.concat(balanced_frames, ignore_index=True)

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print("\n--- Final label distribution ---")
print(df_balanced["label"].value_counts())
print(f"\nOverall balance ratio: {df_balanced['label'].value_counts(normalize=True).round(3).to_dict()}")
print("\n--- Per-language balanced counts ---")
print(df_balanced.groupby(["language", "label"]).size().unstack(fill_value=0))

Balancing classes per language...

ms: scam=3,124 | ham undersampled 4,974 -> 3,124
singlish: scam=3,704 | ham undersampled 47,957 -> 3,704
en: scam=49,155 | ham undersampled 58,765 -> 49,155
ta: scam=4,761 | ham undersampled 4,997 -> 4,761
zh: scam=3,823 | ham undersampled 30,862 -> 3,823

Balanced dataset shape: (129134, 4)

--- Final label distribution ---
label
0    64567
1    64567
Name: count, dtype: int64

Overall balance ratio: {0: 0.5, 1: 0.5}

--- Per-language balanced counts ---
label         0      1
language              
en        49155  49155
ms         3124   3124
singlish   3704   3704
ta         4761   4761
zh         3823   3823


In [8]:
# =============================================================================
# CELL 6: Stratified train / val / test split (70 / 15 / 15)
# -----------------------------------------------------------------------------
# Stratification is on a combined key of (label, language) to ensure each
# language-label combination is proportionally represented in all three splits.
#
# Split procedure:
#   Step 1: split into train (70%) and temp (30%) stratified by label+language
#   Step 2: split temp into val (15%) and test (15%) stratified by label+language
# =============================================================================

# Create combined stratification key
df_balanced["strat_key"] = (
    df_balanced["label"].astype(str) + "_" + df_balanced["language"]
)

# Filter out strat_key groups with fewer than 2 samples (can't stratify)
key_counts = df_balanced["strat_key"].value_counts()
valid_keys = key_counts[key_counts >= 2].index
df_balanced = df_balanced[df_balanced["strat_key"].isin(valid_keys)].reset_index(drop=True)

# Step 1: train vs temp
df_train, df_temp = train_test_split(
    df_balanced,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df_balanced["strat_key"],
    random_state=RANDOM_STATE,
)

# Step 2: val vs test (split temp 50/50 to get 15/15)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    stratify=df_temp["strat_key"],
    random_state=RANDOM_STATE,
)

# Drop the helper column
for df in [df_train, df_val, df_test]:
    df.drop(columns=["strat_key"], inplace=True)

print("--- Split sizes ---")
total = len(df_balanced)
print(f"Train: {len(df_train):,} rows ({len(df_train)/total*100:.1f}%)")
print(f"Val:   {len(df_val):,} rows ({len(df_val)/total*100:.1f}%)")
print(f"Test:  {len(df_test):,} rows ({len(df_test)/total*100:.1f}%)")

print("\n--- Label balance per split ---")
for name, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    counts = df["label"].value_counts(normalize=True).round(3)
    print(f"  {name}: scam={counts.get(1, 0):.3f}  ham={counts.get(0, 0):.3f}")

print("\n--- Language distribution in train ---")
print(df_train["language"].value_counts())

--- Split sizes ---
Train: 90,393 rows (70.0%)
Val:   19,370 rows (15.0%)
Test:  19,371 rows (15.0%)

--- Label balance per split ---
  Train: scam=0.500  ham=0.500
  Val: scam=0.500  ham=0.500
  Test: scam=0.500  ham=0.500

--- Language distribution in train ---
language
en          68816
ta           6665
zh           5352
singlish     5186
ms           4374
Name: count, dtype: int64


In [9]:
# =============================================================================
# CELL 7: Save splits and final verification
# -----------------------------------------------------------------------------
# Saves train.csv, val.csv, test.csv to data/processed/.
# Also saves the full balanced dataset as scamsense_full_dataset.csv
# for reference and EDA in notebook 04.
#
# All files saved with utf-8-sig encoding for Excel compatibility.
# =============================================================================

df_train.to_csv(f"{PROCESSED}/train.csv",    index=False, encoding="utf-8-sig")
df_val.to_csv(  f"{PROCESSED}/val.csv",      index=False, encoding="utf-8-sig")
df_test.to_csv( f"{PROCESSED}/test.csv",     index=False, encoding="utf-8-sig")
df_balanced.to_csv(f"{PROCESSED}/scamsense_full_dataset.csv", index=False, encoding="utf-8-sig")

print("Saved files:")
for fname in ["train.csv", "val.csv", "test.csv", "scamsense_full_dataset.csv"]:
    fpath = f"{PROCESSED}/{fname}"
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  ✓ {fname} ({size_mb:.1f} MB)")

# Final verification — reload train and confirm schema
print("\n--- Final verification (reloading train.csv) ---")
df_check = pd.read_csv(f"{PROCESSED}/train.csv", encoding="utf-8-sig")
print(f"Shape:   {df_check.shape}")
print(f"Columns: {df_check.columns.tolist()}")
print(f"Labels:  {df_check['label'].value_counts().to_dict()}")
print(f"Languages:\n{df_check['language'].value_counts()}")
print("\nSample rows:")
print(df_check.sample(5, random_state=42)[["text", "label", "language", "source"]].to_string(index=False))

print("\n✅ Notebook 03 complete. Proceed to notebook 04 (EDA).")

Saved files:
  ✓ train.csv (134.5 MB)
  ✓ val.csv (27.5 MB)
  ✓ test.csv (27.5 MB)
  ✓ scamsense_full_dataset.csv (190.2 MB)

--- Final verification (reloading train.csv) ---
Shape:   (90393, 4)
Columns: ['text', 'label', 'language', 'source']
Labels:  {1: 45197, 0: 45196}
Languages:
language
en          68816
ta           6665
zh           5352
singlish     5186
ms           4374
Name: count, dtype: int64

Sample rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                